## Setup

The setup cells install every library the pipeline needs at a fixed version, log in to Hugging Face once for the whole session (the token is read from the Kaggle secret `HF_TOKEN`), set the input clip and the output folder, erase any results that earlier runs saved for the clip, create a fresh output folder, copy the input clip into it as the source audio `source.wav`, and record its duration, sample rate and channel count in `metadata.json`. Each stage imports the libraries it needs, reads the outputs of earlier stages from their JSON files, writes its own output to a JSON file named after the stage (for example `alignment.json`), and records the library versions it used in `metadata.json`, together with the settings of the rule-based steps (alignment excerpts and prosody marking). Running a stage again overwrites its previous results. The models, their fixed commits and their generation settings are defined in the stage code and are not repeated in `metadata.json`; each model is loaded at a fixed commit of its Hugging Face repository, so that a later update of the model cannot change the results.

**Parameters:** `AUDIO_PATH` is the Bengali speech clip to translate; `OUTPUT_ROOT` is the folder in which the per-clip results are written.

In [ ]:
# install every library the pipeline needs, at the versions of the reported run,
# before anything is imported
!pip install -q \
    "torch==2.11.0" \
    "torchaudio==2.11.0" \
    "transformers==5.17.0" \
    "huggingface_hub==1.33.0" \
    "uroman==1.3.1.1"

In [ ]:
# log in to Hugging Face once, with the token stored in the Kaggle secret HF_TOKEN
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# the input clip
AUDIO_PATH = "/kaggle/input/..."
# results go to OUTPUT_ROOT/<clip name>/
OUTPUT_ROOT = "/kaggle/working/results"

In [ ]:
# libraries used by the setup cells and the shared helpers
import json
import re
import shutil
from importlib.metadata import version
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

In [ ]:
# check the input clip and create its output folder
AUDIO_PATH = Path(AUDIO_PATH)
OUTPUT_ROOT = Path(OUTPUT_ROOT)
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(AUDIO_PATH)

# the clip's file name without its extension, e.g. 'sample'
CLIP_ID = AUDIO_PATH.stem
OUT_DIR = OUTPUT_ROOT / CLIP_ID
if OUT_DIR.resolve() in AUDIO_PATH.resolve().parents:
    raise ValueError("AUDIO_PATH must not lie inside the clip's output folder, which is erased on every run")

# a new run on the clip erases everything that earlier runs saved for it
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

# the source audio: every stage reads the clip from here
SOURCE = OUT_DIR / "source.wav"
shutil.copy2(AUDIO_PATH, SOURCE)

In [ ]:
# helpers shared by all stages
META_PATH = OUT_DIR / "metadata.json"


def load_meta():
    """Read metadata.json."""
    return json.loads(META_PATH.read_text(encoding="utf-8"))


def save_meta(meta):
    """Write metadata.json."""
    text = json.dumps(meta, ensure_ascii=False, indent=2)
    META_PATH.write_text(text, encoding="utf-8")


def load_output(stage):
    """Read the output of a stage from <stage>.json."""
    return json.loads((OUT_DIR / f"{stage}.json").read_text(encoding="utf-8"))


def save_output(stage, data):
    """Write the output of a stage to <stage>.json."""
    text = json.dumps(data, ensure_ascii=False, indent=4)
    (OUT_DIR / f"{stage}.json").write_text(text, encoding="utf-8")


def record_env(meta, *packages):
    """Add the GPU name and the versions of the given packages to the metadata."""
    import torch

    env = meta.setdefault("environment", {})
    if torch.cuda.is_available():
        env["gpu"] = torch.cuda.get_device_name(0)
    else:
        env["gpu"] = "cpu"
    for pkg in packages:
        env[pkg] = version(pkg)


def free_gpu():
    """Release cached GPU memory after a stage has deleted its model."""
    import gc

    import torch

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# the source audio
source, source_sr = sf.read(str(SOURCE), always_2d=True)
source_channels = source.shape[1]
source = source.mean(axis=1)

meta = {}
meta["clip_id"] = CLIP_ID
meta["input_path"] = str(AUDIO_PATH)
meta["source"] = {
    "duration_s": round(len(source) / source_sr, 3),
    "sample_rate": source_sr,
    "channels": source_channels,
}

save_meta(meta)
display(Audio(str(SOURCE)))

## 1. Transcription and Romanization

The Bengali clip is transcribed into native-script text by the transcription model. The output contains no punctuation. Each word is then romanized with `uroman`, and every character outside the aligner's vocabulary (the Latin letters a-z and the apostrophe) is removed. The romanized words are used only for alignment; later stages refer back to the native-script words.

**Input:** the source audio.

**Output:** `transcription.json` (native-script words and their romanizations).

In [ ]:
import sys

import pandas as pd
import uroman as ur
from huggingface_hub import snapshot_download

pd.set_option("display.max_colwidth", None)

In [ ]:
meta = load_meta()

# the transcription model
ASR_MODEL = "bodhan-ai/indic-transcribe-core"
ASR_REVISION = "e083ab7dfeeddcb5c0b4a2d7df53f53e7b0f1537"  # fixed commit of the model
asr_model_dir = snapshot_download(ASR_MODEL, revision=ASR_REVISION)
sys.path.insert(0, asr_model_dir)

from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(asr_model_dir)
uroman = ur.Uroman()

# Bengali speech to Bengali text
transcription = asr(str(SOURCE), lang="bn")
native_words = transcription.split()

# romanize for the aligner, one romanized word per native word;
# the aligner only knows a-z and the apostrophe, so everything else is removed.
roman_raw = [uroman.romanize_string(w, lcode="ben") for w in native_words]
roman_words = [re.sub(r"[^a-z']", "", r.lower()) for r in roman_raw]

save_output("transcription", {
    "native_words": native_words,
    "roman_words": roman_words,
})
record_env(meta, "torch", "transformers", "huggingface_hub", "uroman")
save_meta(meta)

summary = {
    "transcription": transcription,
    "romanized": " ".join(roman_words),
}
display(pd.DataFrame([summary]))

del asr
free_gpu()

## 2. Forced Alignment

The audio is mixed to mono and resampled to 16 kHz, and the multilingual CTC aligner assigns a start and an end time to every romanized word. Each word also receives a confidence score. A short padded excerpt of every word is saved so that its boundaries can be checked by ear. Words that romanize to nothing (for example stray symbols) cannot be aligned; they are left out of this and all later stages and listed as dropped.

**Input:** the source audio and the romanized words from stage 1.

**Output:** `alignment.json` (per-word timings and scores), `words/NN_<word>.wav` & `alignment` settings in `metadata.json`.

In [ ]:
import pandas as pd
import torch
import torchaudio
from IPython.display import HTML
from torchaudio.pipelines import MMS_FA as bundle

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 1
meta = load_meta()
transcription = load_output("transcription")
native_words = transcription["native_words"]
roman_words = transcription["roman_words"]

# words that romanize to nothing cannot be aligned and are left out
kept = [i for i, r in enumerate(roman_words) if r]
dropped = [native_words[i] for i, r in enumerate(roman_words) if not r]

In [ ]:
# word excerpts
PAD = 0.05  # padding (s) around each saved word excerpt
SR = bundle.sample_rate  # the aligner works at 16 kHz

device = "cuda" if torch.cuda.is_available() else "cpu"
alignmodel = bundle.get_model().to(device)
aligner = bundle.get_aligner()
tokenizer = bundle.get_tokenizer()

# mono 16 kHz audio
waveform, sr = torchaudio.load(str(SOURCE))
waveform = waveform.mean(0, keepdim=True)
waveform = torchaudio.functional.resample(waveform, sr, SR)

with torch.inference_mode():
    # frame-level character probabilities
    emission, _ = alignmodel(waveform.to(device))
    # per word: a list of token spans
    spans = aligner(emission[0], tokenizer([roman_words[k] for k in kept]))

# seconds per emission frame
ratio = waveform.size(1) / emission.size(1) / SR

(OUT_DIR / "words").mkdir(exist_ok=True)
rows = []
excerpts = []
for i, (k, s) in enumerate(zip(kept, spans)):
    w = roman_words[k]
    st = s[0].start * ratio
    en = s[-1].end * ratio
    # confidence, weighted by each token's length in frames
    sc = sum(t.score * len(t) for t in s) / sum(len(t) for t in s)

    # padded excerpt of the word, for checking its boundaries by ear
    excerpt = f"words/{i:02d}_{w}.wav"
    a = max(0, int((st - PAD) * SR))
    b = min(waveform.size(1), int((en + PAD) * SR))
    sf.write(OUT_DIR / excerpt, waveform[0, a:b].numpy(), SR)
    excerpts.append(excerpt)

    row = {
        "#": i,
        "word": w,
        "native": native_words[k],
        "start": round(st, 3),
        "end": round(en, 3),
        "dur": round(en - st, 3),
        "score": round(sc, 2),
    }
    rows.append(row)

scores = [r["score"] for r in rows]
score_mean = round(float(np.mean(scores)), 2)

save_output("alignment", {
    "speech_start": rows[0]["start"],
    "speech_end": rows[-1]["end"],
    "score_mean": score_mean,
    "score_min": min(scores),
    "score_max": max(scores),
    "dropped": dropped,
    "words": rows,
})

meta["alignment"] = {
    "config": {
        "excerpt_pad": PAD,
    },
}
record_env(meta, "torch", "torchaudio")
save_meta(meta)

print(f"Speech: {rows[0]['start']}-{rows[-1]['end']} s")
print(f"Confidence: mean {score_mean}, range {min(scores)}-{max(scores)}")
print(f"Dropped: {', '.join(dropped) or 'none'}")

# word table with a player for each excerpt
players = [Audio(str(OUT_DIR / e))._repr_html_() for e in excerpts]
word_table = pd.DataFrame(rows).assign(audio=players)
display(HTML(word_table.to_html(escape=False, index=False)))

del alignmodel, emission
free_gpu()

## 3. Prosody Marking

The word timings and the waveform are analyzed with fixed rules.

- **Pauses:** the silence before each word is compared with two thresholds. A gap of at least 0.20 s is marked as a short pause (`,`), and a gap of at least 0.50 s as a long pause (`|`). The silences before the first word and after the last word are not marked.
- **Loudness:** each word's level is taken as the mean of the loudest half of its 25 ms frames and compared with the rest of the clip; words well above or below the clip's typical level are labelled *loud* or *soft*.
- **Elongation:** each word's duration is divided by its romanized length, and words that are unusually slow for their length are labelled *dragged*.

The marked sentence is rebuilt with each pause marker placed before the word that follows the pause. Only the pause markers are written into the sentence. For every word, the loudness and elongation measurements and the resulting labels (*loud*, *soft*, *dragged*) are recorded for analysis.

**Input:** the source audio and the word timings from stage 2.

**Output:** `marking.json` (the Bengali transcript without and with pause markers, the pauses, and the measurements and labels of every word) & `marking` settings in `metadata.json`.

In [ ]:
import pandas as pd
import torchaudio
from torchaudio.pipelines import MMS_FA

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 2
meta = load_meta()
rows = load_output("alignment")["words"]
SR = MMS_FA.sample_rate  # the aligner's 16 kHz, as in stage 2

# the same mono 16 kHz audio that the aligner used, and its length in seconds
waveform, sr = torchaudio.load(str(SOURCE))
waveform = waveform.mean(0, keepdim=True)
waveform = torchaudio.functional.resample(waveform, sr, SR)
total = waveform.size(1) / SR

In [ ]:
# pauses
SHORT_TOK = ","
LONG_TOK = "|"
SHORT_TH = 0.20  # a gap of at least this (s) is a short pause
LONG_TH = 0.50  # a gap of at least this (s) is a long pause
MARK_EDGES = False  # also mark the silence before the first and after the last word

# loudness
EMPH_Z = 0.8  # |z| at or above this: the word is "loud" / "soft"
FRAME = 0.025  # analysis frame length (s)
HOP = 0.010  # hop between frames (s)

# elongated words
DRAG_Z = 1.0  # robust z-score at or above this: the word is "dragged"
MIN_CHARS = 2  # shorter words are skipped, as their per-character timing is noisy

# sentence markers, e.g. "^", "~", "..."
LOUD_MARK = ""
SOFT_MARK = ""
DRAG_MARK = ""

In [ ]:
# per-word data from the alignment
wav = waveform.squeeze().numpy()
N = len(rows)
words = [r["native"] for r in rows]
romans = [r["word"] for r in rows]
start = np.array([r["start"] for r in rows])
end = np.array([r["end"] for r in rows])
dur = end - start

# gap before each word: index 0 is the leading silence, and overlaps count as 0
gap_before = np.concatenate([[start[0]], start[1:] - end[:-1]])
gap_before = np.clip(gap_before, 0, None)
gap_after_last = max(total - end[-1], 0)



def pause_token(gap):
    """Pause marker for a silence of the given length."""
    if gap >= LONG_TH:
        return LONG_TOK
    if gap >= SHORT_TH:
        return SHORT_TOK
    return ""


pause = [pause_token(g) for g in gap_before]
tail = pause_token(gap_after_last)
if not MARK_EDGES:
    pause[0] = ""
    tail = ""

# every marked pause as a span of silence
# ("before" is the word it precedes, "end" is the trailing silence)
pause_spans = []
for i in range(N):
    if pause[i]:
        span_start = end[i - 1] if i > 0 else 0.0
        pause_spans.append((words[i], span_start, start[i], pause[i]))
if tail:
    pause_spans.append(("end", end[-1], total, tail))

pauses = []
for w, a, b, tok in pause_spans:
    span = {
        "before": w,
        "start": round(float(a), 3),
        "end": round(float(b), 3),
        "length": round(float(b - a), 3),
        "type": "long" if tok == LONG_TOK else "short",
    }
    pauses.append(span)

In [ ]:
# frame levels in dB
frame_len = int(FRAME * SR)
hop_len = int(HOP * SR)
frames = np.lib.stride_tricks.sliding_window_view(wav, frame_len)[::hop_len]
frame_rms = np.sqrt((frames**2).mean(axis=1) + 1e-10)
db = 20 * np.log10(frame_rms + 1e-10)

# per word: the mean of the loudest half of its frames, so that stop-consonant dips are ignored
loud_db = []
for s, e in zip(start, end):
    a = min(int(s / HOP), len(db) - 1)
    b = min(max(int(e / HOP), a + 1), len(db))
    seg = np.sort(db[a:b])
    loud_db.append(seg[len(seg) // 2 :].mean())
loud_db = np.array(loud_db)

# level relative to the clip's average word, in dB
db_vs_mean = loud_db - loud_db.mean()

# words well above / below the clip's average are "loud" / "soft"
loud_z = db_vs_mean / (loud_db.std() + 1e-9)
emph = np.where(loud_z <= -EMPH_Z, "soft", "")
emph = np.where(loud_z >= EMPH_Z, "loud", emph)

In [ ]:
# seconds per romanized character, compared with the clip's median by a robust (median / MAD) z-score
nchar = np.array([len(r) for r in romans])
spc = dur / np.maximum(nchar, 1)
med = np.median(spc)
mad = np.median(np.abs(spc - med)) * 1.4826
drag_z = (spc - med) / (mad + 1e-9)

# duration relative to the duration expected for the word's length (2.0 = twice as long)
x_expected = dur / (np.maximum(nchar, 1) * med + 1e-9)
drag = (drag_z >= DRAG_Z) & (nchar >= MIN_CHARS)

In [ ]:
# markers placed before / after each word
pre = []
for e in emph:
    if e == "loud":
        pre.append(LOUD_MARK)
    elif e == "soft":
        pre.append(SOFT_MARK)
    else:
        pre.append("")
suf = [DRAG_MARK if d else "" for d in drag]

# marked sentences, and the measurements and labels of every word
bn_parts = []
roman_parts = []
word_info = []
for i in range(N):
    marked = pre[i] + words[i] + suf[i]
    bn_parts += [pause[i], marked]
    roman_parts += [pause[i], pre[i] + romans[i] + suf[i]]
    labels = []
    if emph[i]:
        labels.append(str(emph[i]))
    if drag[i]:
        labels.append("dragged")
    info = {
        "#": i,
        "word": words[i],
        "loud_z": round(float(loud_z[i]), 2),
        "db_vs_mean": round(float(db_vs_mean[i]), 1),
        "drag_z": round(float(drag_z[i]), 2),
        "x_expected": round(float(x_expected[i]), 2),
        "labels": labels,
    }
    word_info.append(info)
bn_parts.append(tail)
roman_parts.append(tail)

bn_clean = " ".join(words)
bn_marked = " ".join(p for p in bn_parts if p)
roman_marked = " ".join(p for p in roman_parts if p)

save_output("marking", {
    "text": bn_clean,
    "text_marked": bn_marked,
    "roman_marked": roman_marked,
    "pauses": pauses,
    "words": word_info,
})

meta["marking"] = {
    "config": {
        "short_tok": SHORT_TOK,
        "long_tok": LONG_TOK,
        "short_th": SHORT_TH,
        "long_th": LONG_TH,
        "mark_edges": MARK_EDGES,
        "emph_z": EMPH_Z,
        "frame": FRAME,
        "hop": HOP,
        "drag_z": DRAG_Z,
        "min_chars": MIN_CHARS,
        "loud_mark": LOUD_MARK,
        "soft_mark": SOFT_MARK,
        "drag_mark": DRAG_MARK,
    },
}
save_meta(meta)

print(f"Pause thresholds: short {SHORT_TH:.2f} s, long {LONG_TH:.2f} s")
# the saved measurements, with the gap and pause marker before each word
word_table = pd.DataFrame(word_info)
word_table.insert(2, "gap_before", np.round(gap_before, 3))
word_table.insert(3, "pause", pause)
display(word_table)
display(pd.DataFrame(pauses))

sentences = {"Bengali (marked)": bn_marked, "Roman (marked)": roman_marked}
display(pd.DataFrame([sentences]))

## 4. Translation

The translation model translates two versions of the Bengali sentence into English: the clean transcript and the pause-marked transcript. Decoding is greedy, so the same input always produces the same output.

**Input:** the clean and the pause-marked Bengali sentence from stage 3.

**Output:** `translation.json` (the English translation without and with pause markers, and the markers carried over).

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 3
meta = load_meta()
marking = load_output("marking")
bn_clean = marking["text"]
bn_marked = marking["text_marked"]
SHORT_TOK = meta["marking"]["config"]["short_tok"]
LONG_TOK = meta["marking"]["config"]["long_tok"]

In [ ]:
MT_MODEL = "bodhan-ai/indic-translate"
MT_REVISION = "2dc0f1d00fc3b5f84a93052b0c55a415bfd3adb9"  # fixed commit of the model
PROMPT = "Translate the following text into {target_language}:\n\n{text}"
MAX_NEW_TOKENS = 512

transprocessor = AutoProcessor.from_pretrained(MT_MODEL, revision=MT_REVISION)
transmodel, mt_info = AutoModelForImageTextToText.from_pretrained(
    MT_MODEL,
    revision=MT_REVISION,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
    output_loading_info=True,
)
transmodel.eval()

# loading problems (missing or unexpected weights)
mt_loading = {k: sorted(map(str, v)) for k, v in mt_info.items() if v}
if mt_loading:
    print("WARNING: problems loading the translation model:", mt_loading)


def translate(text, target_language="English", max_new_tokens=MAX_NEW_TOKENS):
    """Translate text with greedy decoding."""
    prompt = PROMPT.format(target_language=target_language, text=text)
    messages = [{"role": "user", "content": prompt}]
    inputs = transprocessor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(transmodel.device)

    with torch.inference_mode():
        output = transmodel.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    # keep only the newly generated tokens
    new_tokens = output[0, inputs["input_ids"].shape[-1] :]
    return transprocessor.decode(new_tokens, skip_special_tokens=True).strip()


en_clean = translate(bn_clean)
en_marked = translate(bn_marked)

save_output("translation", {
    "text": en_clean,
    "text_marked": en_marked,
    # does the translator carry the pause markers into its output?
    "markers_in": {
        "long": bn_marked.count(LONG_TOK),
        "short": bn_marked.count(SHORT_TOK),
    },
    "markers_out": {
        "long": en_marked.count(LONG_TOK),
        "short": en_marked.count(SHORT_TOK),
    },
})
record_env(meta, "torch", "transformers")
save_meta(meta)

comparison = [
    {"": "clean", "Bengali": bn_clean, "English": en_clean},
    {"": "marked", "Bengali": bn_marked, "English": en_marked},
]
display(pd.DataFrame(comparison))

del transmodel, transprocessor
free_gpu()

## 5. Voice-Cloned Speech Synthesis

Both English translations are spoken by a zero-shot voice-cloning speech synthesis model. The source audio serves as the voice reference for each. The clean translation is spoken with the clean Bengali transcript as the transcript of the reference, which gives the translated audio; the pause-marked translation is spoken with the pause-marked Bengali transcript, which gives the marked translated audio. Comparing the two shows the effect of the pause markers on the synthesized speech. Each generation samples with the same fixed seed, so a run can be repeated on the same hardware.

**Input:** the source audio, the clean and marked Bengali transcripts from stage 3 and the clean and marked English translations from stage 4.

**Output:** `translated.wav` and `translated_marked.wav` (the clean and the marked translation, spoken) & `synthesis.json` (their durations).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# inputs from stages 3 and 4
meta = load_meta()
marking = load_output("marking")
translation = load_output("translation")
bn_clean = marking["text"]
bn_marked = marking["text_marked"]
en_clean = translation["text"]
en_marked = translation["text_marked"]

# voice reference: the source audio, mono, at its own sample rate
source, source_sr = sf.read(str(SOURCE), always_2d=True)
source = source.mean(axis=1)

In [ ]:
TTS_MODEL = "multimodalart/higgs-audio-v3-tts-4b-transformers"
TTS_REVISION = "30f01593ee6a12efa586c92455afe4b76e45095d"  # fixed commit of the model and its code
TEMPERATURE = 0.8
TOP_K = 50
SEED = 0  # fixes the sampling, so that a run can be repeated on the same hardware
sample_rate = 24_000
device = "cuda:0" if torch.cuda.is_available() else "cpu"

ttstokenizer = AutoTokenizer.from_pretrained(
    TTS_MODEL,
    revision=TTS_REVISION,
    trust_remote_code=True,
)
ttsmodel, tts_info = AutoModelForCausalLM.from_pretrained(
    TTS_MODEL,
    revision=TTS_REVISION,
    dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
    output_loading_info=True,
)

# weights missing from the checkpoint are randomly initialised
tts_loading = {k: sorted(map(str, v)) for k, v in tts_info.items() if v}
if tts_loading.get("missing_keys"):
    print("WARNING: weights missing from the checkpoint:", tts_loading["missing_keys"])

reference_audio = torch.tensor(source, dtype=torch.float32).to(device)
source_s = meta["source"]["duration_s"]


def synthesize(text, reference_text, filename):
    """Speak the text in the voice of the source audio, save it, and describe the result."""
    # the same seed for every output, so that each can be repeated on its own
    torch.manual_seed(SEED)
    speech = ttsmodel.generate_speech(
        text,
        ttstokenizer,
        reference_audio=reference_audio,
        reference_sample_rate=source_sr,
        reference_text=reference_text,
        temperature=TEMPERATURE,
        top_k=TOP_K,
    )
    audio = speech.detach().float().cpu().numpy().squeeze()
    if audio.size == 0:
        raise RuntimeError(f"TTS returned empty audio for {filename}")

    sf.write(OUT_DIR / filename, audio, sample_rate)
    duration_s = round(audio.size / sample_rate, 3)
    display(Audio(audio, rate=sample_rate))
    return {
        "file": filename,
        "duration_s": duration_s,
        # negative: the translated audio is shorter than the source audio
        "length_change_pct": round(100 * (duration_s - source_s) / source_s, 1),
    }


# clean translation with the clean transcript, marked translation with the marked transcript
print("Translated:")
translated = synthesize(en_clean, bn_clean, "translated.wav")
print("Translated (marked):")
translated_marked = synthesize(en_marked, bn_marked, "translated_marked.wav")

save_output("synthesis", {
    "translated": translated,
    "translated_marked": translated_marked,
})
record_env(meta, "torch", "transformers")
save_meta(meta)

del ttsmodel, ttstokenizer, reference_audio
free_gpu()

## Results and Archive

Once every stage has completed, the main figures of the run are summarized, the source audio and both translated audios are presented for listening, and the clip's output folder is compressed into a single archive.

In [ ]:
import zipfile

import pandas as pd

pd.set_option("display.max_colwidth", None)

In [ ]:
meta = load_meta()
al = load_output("alignment")
mk = load_output("marking")
tr = load_output("translation")
tt = load_output("synthesis")

speech_span = f"{al['speech_start']} - {al['speech_end']}"
confidence = f"{al['score_mean']} ({al['score_min']} - {al['score_max']})"
mk_config = meta["marking"]["config"]
thresholds = f"{mk_config['short_th']} / {mk_config['long_th']}"
n_long = sum(p["type"] == "long" for p in mk["pauses"])
n_short = sum(p["type"] == "short" for p in mk["pauses"])
n_label = {
    label: sum(label in w["labels"] for w in mk["words"])
    for label in ("loud", "soft", "dragged")
}
prosody = f"{n_label['loud']} / {n_label['soft']} / {n_label['dragged']}"
markers = f"{tr['markers_in']['long']} / {tr['markers_out']['long']}"


def duration(output):
    """Duration of a translated audio, with its change relative to the source audio."""
    return f"{output['duration_s']} ({output['length_change_pct']:+.1f}%)"


summary = [
    ("source duration (s)", meta["source"]["duration_s"]),
    ("words", len(load_output("transcription")["native_words"])),
    ("speech span (s)", speech_span),
    ("confidence mean (range)", confidence),
    ("pause thresholds short / long (s)", thresholds),
    ("pauses (long / short)", f"{n_long} / {n_short}"),
    ("loud / soft / dragged words", prosody),
    ("long-pause markers in Bengali / English", markers),
    ("translated duration (s)", duration(tt["translated"])),
    ("translated duration, marked (s)", duration(tt["translated_marked"])),
    ("Bengali (marked)", mk["text_marked"]),
    ("Roman (marked)", mk["roman_marked"]),
    ("English (clean)", tr["text"]),
    ("English (marked)", tr["text_marked"]),
]
display(pd.DataFrame(summary, columns=["", CLIP_ID]))

print("Source:")
display(Audio(str(SOURCE)))
print("Translated:")
display(Audio(str(OUT_DIR / "translated.wav")))
print("Translated (marked):")
display(Audio(str(OUT_DIR / "translated_marked.wav")))

archive_path = OUTPUT_ROOT.parent / f"working/{CLIP_ID}/package.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_ROOT.parent))

print(archive_path)